In [ ]:
from nav2_simple_commander.robot_navigator import BasicNavigator
import rclpy

In [2]:
rclpy.init()
nav = BasicNavigator()

In [3]:
nav.waitUntilNav2Active()

[INFO] [1793928742.852418631] [basic_navigator]: Nav2 is ready for use!


In [4]:
import math
import tf_transformations

def get_quaternion_from_yaw(yaw_degrees):
    yaw_radians = math.radians(yaw_degrees)
    quaternion = tf_transformations.quaternion_from_euler(0, 0, yaw_radians)
    return quaternion

In [5]:
sub_amcl = rclpy.create_node('sub_amcl')

In [6]:
def amcl_callback(msg):
    print("===")
    print(msg.pose.pose.position)
    print(msg.pose.pose.orientation)
    

In [7]:
from geometry_msgs.msg import PoseWithCovarianceStamped

In [8]:
sub_amcl.create_subscription(PoseWithCovarianceStamped, 'amcl_pose', amcl_callback,10)

In [9]:
from geometry_msgs.msg import PoseStamped

In [10]:
goal_yaw = 180
q = get_quaternion_from_yaw(goal_yaw)

goal_pose=PoseStamped()
goal_pose.header.frame_id = 'map'
goal_pose.header.stamp = nav.get_clock().now().to_msg()
goal_pose.pose.position.x = 0.5
goal_pose.pose.position.y = 0.0
goal_pose.pose.position.z = 0.0
goal_pose.pose.orientation.x=q[0]
goal_pose.pose.orientation.y=q[1]
goal_pose.pose.orientation.z=q[2]
goal_pose.pose.orientation.w=q[3]

In [11]:
nav.goToPose(goal_pose)

[INFO] [1793928765.757070540] [basic_navigator]: Navigating to goal: 0.5 0.0...


True

In [12]:
from geometry_msgs.msg import PoseStamped

goal_yaw = 90
q = get_quaternion_from_yaw(goal_yaw)
goal_pose = PoseStamped()
goal_pose.header.frame_id = 'map'
goal_pose.header.stamp = nav.get_clock().now().to_msg()
goal_pose.pose.position.x=7.79
goal_pose.pose.position.y=0.12
goal_pose.pose.position.z=-0.00
goal_pose.pose.orientation.x=q[0]
goal_pose.pose.orientation.y=q[1]
goal_pose.pose.orientation.z=q[2]
goal_pose.pose.orientation.w=q[3]

In [13]:
nav.goToPose(goal_pose)

[INFO] [1793928818.664614059] [basic_navigator]: Navigating to goal: 7.79 0.12...


True

In [ ]:
from rclpy.duration import Duration 
from IPython.display import clear_output

while not nav.isTaskComplete():
    clear_output(wait=True)
    
    rclpy.spin_once(sub_amcl, timeout_sec=1.0)

    feedback = nav.getFeedback()
    print("===")
    
    print('Distance remaining: ' + '{:.2f}'.format(feedback.distance_remaining) + ' meters.')

    
    if Duration.from_msg(feedback.navigation_time) > Duration(seconds=30.0):
        nav.cancelTask()

        

TypeError: spin_once() got an unexpected keyword argument 'timeout_set'

In [16]:
from nav2_simple_commander.robot_navigator import TaskResult
result = nav.getResult()
if result == TaskResult.SUCCEEDED:
    print('Goal succeeded!')
elif result == TaskResult.CANCELED:
    print('Goal was canceled!')
elif result == TaskResult.FAILED:
    print('Goal failed!')
else:
    print('Goal has an invalid return status!')
    

Goal has an invalid return status!


In [17]:
goal_pose_list = []
goal_yaw1 =0
q1 = get_quaternion_from_yaw(goal_yaw1)
goal_pose1 = PoseStamped()
goal_pose1.header.frame_id = 'map'
goal_pose1.header.stamp=nav.get_clock().now().to_msg()
goal_pose1.pose.position.x= 1.658
goal_pose1.pose.position.y=-0.010
goal_pose1.pose.orientation.x=q[0]
goal_pose1.pose.orientation.y=q[1]
goal_pose1.pose.orientation.z=q[2]
goal_pose1.pose.orientation.w=q[2]
goal_pose_list.append(goal_pose1)

goal_yaw2=180
q2 = get_quaternion_from_yaw(goal_yaw2)
goal_pose2 = PoseStamped()
goal_pose2.header.frame_id = 'map'
goal_pose2.header.stamp=nav.get_clock().now().to_msg()
goal_pose2.pose.position.x=3.7301
goal_pose2.pose.position.y=0.0358
goal_pose2.pose.orientation.x=q[0]
goal_pose2.pose.orientation.y=q[1]
goal_pose2.pose.orientation.z=q[2]
goal_pose2.pose.orientation.w=q[2]
goal_pose_list.append(goal_pose2)

goal_yaw3=45
q3 = get_quaternion_from_yaw(goal_yaw3)
goal_pose3 = PoseStamped()
goal_pose3.header.frame_id = 'map'
goal_pose3.header.stamp=nav.get_clock().now().to_msg()
goal_pose3.pose.position.x=5.8468
goal_pose3.pose.position.y=0.1043
goal_pose3.pose.orientation.x=q[0]
goal_pose3.pose.orientation.y=q[1]
goal_pose3.pose.orientation.z=q[2]
goal_pose3.pose.orientation.w=q[2]
goal_pose_list.append(goal_pose3)


In [18]:
nav_start=nav.get_clock().now()
nav.followWaypoints(goal_pose_list)


[INFO] [1793928876.626011049] [basic_navigator]: Following 3 goals....


True

In [19]:
from rclpy.duration import Duration

while not nav.isTaskComplete():
    clear_output(wait=True)
    rclpy.spin_once(sub_amcl, timeout_sec=1.0)
    
    feedback = nav.getFeedback()
    print("===")
    print('Executing current waypoint: ' + str(feedback.current_waypoint + 1) + '/' + str(len(goal_pose_list)))
    
    now = nav.get_clock().now()
    # 💡 Duration 생성 시 키워드 인자(seconds=)를 빼고 정수 혹은 나노초로 전달하며, navigator를 nav로 수정합니다.
    if now - nav_start > Duration(seconds=300): 
        nav.cancelTask()


===
Executing current waypoint: 3/3


In [20]:
nav.lifecycleShutdown()
rclpy.shutdown()

[INFO] [1793928968.990831612] [basic_navigator]: Shutting down lifecycle nodes based on lifecycle_manager.
[INFO] [1793928968.999212042] [basic_navigator]: Shutting down /lifecycle_manager_localization/manage_nodes
[INFO] [1793928969.403856024] [basic_navigator]: Shutting down /lifecycle_manager_navigation/manage_nodes
